# Paper figure — DLS by prefix length (all datasets)

Standalone, **read-only** figure notebook. It reloads the cached evaluation results
(`eval_results/.../*_outputs.pkl`, produced by the `pipeline_*.ipynb` notebooks with
`RUN_EVAL=True`) and renders one combined figure for the LaTeX paper:

* **grid**: rows = datasets, columns = models;
* per panel: four **DLS-by-prefix-length** curves, one colour each —
  `clean` (baseline), `decision_train`, `decision_decoding`,
  `decision_train_decode`;
* a single **shared legend** (one horizontal row, below the grid) explains the
  colours, so individual panels stay uncluttered;
* a grey **prefix-length density** "mountain" behind the curves (how many test
  prefixes exist at each prefix length).

Output is written as **`.pgf`** (for `\input{}` into LaTeX) and **`.pdf`**
(for `\includegraphics`). Nothing is re-decoded — it only reads the cache.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_THREADS", "1")

import sys
import importlib
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

sys.path.insert(0, "..")  # this notebook: src/notebooks/, package: src/suffix_pred/

import suffix_pred.experiments as exp
import suffix_pred.experiments.evaluation as evaluation
for m in (exp, evaluation):
    importlib.reload(m)
from suffix_pred.experiments import make_experiment, DATASETS, MODELS, Variant

# Which datasets / models to lay out (rows x columns of the grid).
PLOT_DATASETS = ["Helpdesk", "Sepsis", "BPIC20_DD", "Procurement"]
PLOT_MODELS   = ["FS", "GAN", "UED"]

# The four variants compared in every panel:
# (variant, label, colour, marker, linestyle).
# Each variant has its own colour AND linestyle AND marker, so overlapping,
# near-identical curves can still be told apart (solid vs dashed vs dotted ...).
CURVES = [
    ("clean",                 "Clean (baseline)",  "tab:blue",   "x", "-"),
    ("decision_train",        "Dec. train",        "tab:green",  "s", "--"),
    ("decision_decoding",     "Dec. decode",        "tab:orange", "^", "-."),
    ("decision_train_decode", "Dec. train+decode", "tab:red",    "o", ":"),
]

# Pretty names for titles / labels.
DATASET_LABELS = {"Helpdesk": "Helpdesk", "Sepsis": "Sepsis",
                  "Procurement": "Procurement", "BPIC20_DD": "BPIC20"}
MODEL_LABELS   = {"UED": "UED-LSTM", "FS": "FS-LSTM", "GAN": "GAN-LSTM"}

# Curve / density shaping.
MAX_PREFIX_LEN   = None   # int to cap the x-axis (drop long sparse tails), or None for all
MIN_SUPPORT      = 1      # drop prefix lengths seen fewer than this many times from the curves
DENSITY_PEAK_FRAC = 1.0   # fraction of panel height the density 'mountain' peak reaches (0..1)
DENSITY_LABEL    = "Prefix-length density"

# Line styling: thin lines + small open markers, staggered so overlaps stay readable.
LINE_WIDTH   = 0.9
MARKER_SIZE  = 3.5
MARKER_EDGE  = 0.8
MARKER_STEP  = len(CURVES)   # markers placed every Nth point, offset per curve -> interleaved

# Where to write the figure (relative to this notebook).
OUT_DIR  = Path("paper_figures")
OUT_STEM = "dls_by_prefix_length_all_datasets"
EXPORT_PGF = True         # also write a .pgf for \input{} into LaTeX

print("datasets:", PLOT_DATASETS)
print("models  :", PLOT_MODELS)
print("curves  :", [c[0] for c in CURVES])

In [ ]:
# ----------------------------------------------- LaTeX-friendly matplotlib style
# Fonts are left to the LaTeX document (pgf.rcfonts=False); text.usetex stays off
# so the inline PDF preview renders without a local LaTeX install. The .pgf file
# still picks up the paper's fonts when compiled.
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "pgf.rcfonts": False,
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 120,
})

## Reload cached results

`evaluation.evaluate(cfg, force=False)` reads the stored `*_outputs.pkl` only and
raises `FileNotFoundError` if a combination was never decoded — those panels are
skipped with a note. The prefix-length **density** is taken from the `clean`
outputs of each (dataset, model); every variant decodes the same test prefixes,
so the density is shared per panel.

In [ ]:
def _per_prefix(dataset, model, variant):
    """Reload one cached result; return (per_prefix_df, avg) or (None, None)."""
    try:
        r = evaluation.evaluate(make_experiment(dataset, model, variant), force=False)
    except Exception as e:
        print(f"  skip {dataset}/{model}/{variant}: {type(e).__name__}: {str(e)[:80]}")
        return None, None
    return r.per_prefix.copy(), r.avg


def _density(dataset, model, ref_variant="clean"):
    """Number of test prefixes per prefix length, from a reference variant's outputs."""
    try:
        r = evaluation.evaluate(make_experiment(dataset, model, ref_variant), force=False)
    except Exception:
        return None
    counts = (pd.Series([int(o["prefix_len"]) for o in r.outputs])
              .value_counts().sort_index())
    return pd.DataFrame({"prefix_len": counts.index.to_numpy(),
                         "count": counts.to_numpy()})


# results[(dataset, model)] = {"curves": {variant: (df, avg)}, "density": df}
results = {}
for dataset in PLOT_DATASETS:
    for model in PLOT_MODELS:
        print(f"loading {dataset}/{model} ...")
        curves = {}
        for variant, *_ in CURVES:
            df, avg = _per_prefix(dataset, model, variant)
            if df is not None:
                curves[variant] = (df, avg)
        results[(dataset, model)] = {
            "curves": curves,
            "density": _density(dataset, model, CURVES[0][0]),
        }

n_ok = sum(len(v["curves"]) for v in results.values())
print(f"\nloaded {n_ok} curves across {len(results)} (dataset, model) panels.")

## Build the combined figure

Each panel shares a hidden twin y-axis carrying the grey prefix-length density
(`DENSITY_PEAK_FRAC` controls how tall the “mountain” gets); the DLS curves are
drawn on the primary 0–1 axis on top of it.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

# Open markers (white face) read more clearly than filled ones when curves overlap.
_OPEN_MARKERS = {"o", "s", "^", "v", "D", "d", "p", "*"}


def _filter_len(df, col="prefix_len"):
    if df is None:
        return df
    out = df
    if MAX_PREFIX_LEN is not None:
        out = out[out[col] <= MAX_PREFIX_LEN]
    return out


def _draw_density(ax, density):
    """Grey filled prefix-length 'mountain' on a hidden twin axis, behind curves."""
    if density is None or density.empty:
        return
    d = _filter_len(density)
    axd = ax.twinx()
    axd.fill_between(d["prefix_len"], 0, d["count"],
                     color="0.6", alpha=0.40, linewidth=0, zorder=0)
    axd.plot(d["prefix_len"], d["count"], color="0.5", linewidth=0.5,
             linestyle="--", zorder=0.5)
    peak = max(1, int(d["count"].max()))
    axd.set_ylim(0, peak / max(1e-6, DENSITY_PEAK_FRAC))
    axd.set_yticks([])
    axd.set_zorder(0)
    # keep the DLS axis (and its grid/curves) visually on top of the mountain
    ax.set_zorder(1)
    ax.patch.set_visible(False)


def _draw_curves(ax, curves):
    for idx, (variant, label, colour, marker, ls) in enumerate(CURVES):
        entry = curves.get(variant)
        if entry is None:
            continue
        df, _avg = entry
        df = _filter_len(df)
        if MIN_SUPPORT > 1 and "count" in df.columns:
            df = df[df["count"] >= MIN_SUPPORT]
        face = "white" if marker in _OPEN_MARKERS else colour
        ax.plot(df["prefix_len"], df["dls"],
                color=colour, linewidth=LINE_WIDTH, linestyle=ls,
                marker=marker, markersize=MARKER_SIZE,
                markeredgewidth=MARKER_EDGE, markerfacecolor=face,
                markeredgecolor=colour,
                markevery=(idx, MARKER_STEP),  # interleave markers across curves
                zorder=3)


n_rows, n_cols = len(PLOT_DATASETS), len(PLOT_MODELS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.0 * n_rows),
                         sharex=False, sharey=True, squeeze=False)

for i, dataset in enumerate(PLOT_DATASETS):
    for j, model in enumerate(PLOT_MODELS):
        ax = axes[i][j]
        panel = results.get((dataset, model), {})
        _draw_density(ax, panel.get("density"))
        _draw_curves(ax, panel.get("curves", {}))
        ax.set_ylim(0.0, 1.0)
        ax.margins(x=0.02)
        # prefix length is a positive integer -> force integer-only x ticks
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        if i == 0:
            ax.set_title(MODEL_LABELS.get(model, model))
        if j == 0:
            ax.set_ylabel(f"{DATASET_LABELS.get(dataset, dataset)}\nDLS")
        if i == n_rows - 1:
            ax.set_xlabel("Prefix length")

# Single shared legend explaining the colours/styles (one horizontal row to save space).
handles = [Line2D([0], [0], color=colour, linewidth=1.3, linestyle=ls,
                  marker=marker, markersize=4, markeredgewidth=MARKER_EDGE,
                  markerfacecolor="white" if marker in _OPEN_MARKERS else colour,
                  markeredgecolor=colour, label=label)
           for (_v, label, colour, marker, ls) in CURVES]
handles.append(Patch(facecolor="0.6", alpha=0.40, label=DENSITY_LABEL))
fig.legend(handles=handles, loc="lower center", ncol=len(handles),
           bbox_to_anchor=(0.5, -0.02), frameon=False,
           handlelength=2.2, columnspacing=1.4)

fig.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

## Export for LaTeX

* **PDF** — `\includegraphics{...}` (always written).
* **PGF** — `\input{...}` so the plot inherits the document's fonts/size
  (written when `EXPORT_PGF=True`; the `.pgf` backend itself needs no LaTeX run
  here, but compiling it later does).

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

pdf_path = OUT_DIR / f"{OUT_STEM}.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print("wrote", pdf_path.resolve())

if EXPORT_PGF:
    pgf_path = OUT_DIR / f"{OUT_STEM}.pgf"
    try:
        fig.savefig(pgf_path, bbox_inches="tight")
        print("wrote", pgf_path.resolve())
    except Exception as e:
        print("PGF export skipped:", type(e).__name__, str(e)[:120])

print("\nLaTeX usage:")
print(r"  \includegraphics[width=\linewidth]{%s}" % pdf_path.name)
if EXPORT_PGF:
    print(r"  \input{%s}   %% needs \usepackage{pgf} in the preamble" % (OUT_DIR / f'{OUT_STEM}.pgf'))